(mmm_funnel_from_yml)=
# Building funnel-aware MMMs from YAML

This notebook shows how to declare funnel structure in YAML and build models with
`build_mmm_from_yaml`. It replays the synthetic experiments from
{ref}`mmm_funnel_mueffect` (intro topology) and {ref}`mmm_funnel_mueffect_advanced`
(geo panel) using YAML specs instead of hand-wired Python.

The causal story and DAG reasoning live in those notebooks; here we focus on the
**configuration surface**: `effects:` for `MuEffect` subclasses and `extra_vars`
so mediator columns survive the DataFrame conversion.

```{important}
`FunnelEffect` is defined **in this notebook** (not in the library yet). YAML
references it through a short name registered on the builder `REGISTRY`.
```


In [ ]:
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc.dims as pmd
import pytensor.tensor as pt
import xarray as xr
from pydantic import InstanceOf
from pymc.model.transform.conditioning import do
from pymc_extras.prior import Prior

from pymc_marketing.mmm import (
    AdstockTransformation,
    SaturationTransformation,
)
from pymc_marketing.mmm.additive_effect import (
    DataVarMuEffect,
    IncrementalitySpec,
)
from pymc_marketing.mmm.builders.factories import REGISTRY
from pymc_marketing.mmm.builders.yaml import build_mmm_from_yaml
from pymc_marketing.mmm.media_transformation import MediaTransformation
from pymc_marketing.paths import data_dir

warnings.filterwarnings("ignore")

az.style.use("arviz-darkgrid")
plt.rcParams["figure.figsize"] = [12, 7]
plt.rcParams["figure.dpi"] = 100

rng = np.random.default_rng(42)
CONFIG_DIR = data_dir / "config_files"

In [ ]:
class FunnelEffect(DataVarMuEffect):
    """Intro-topology funnel effect (ported from mmm_funnel_mueffect)."""

    data_vars: list[str] = ["lower_spend", "lower_control"]
    prefix: str = "funnel"
    adstock_uf: InstanceOf[AdstockTransformation]
    saturation_uf: InstanceOf[SaturationTransformation]
    adstock_lf: InstanceOf[AdstockTransformation]
    saturation_lf: InstanceOf[SaturationTransformation]

    def to_dict(self) -> dict:
        """Serialize effect configuration for model persistence."""
        return {
            "prefix": self.prefix,
            "data_vars": list(self.data_vars),
            "adstock_uf": self.adstock_uf.to_dict(),
            "saturation_uf": self.saturation_uf.to_dict(),
            "adstock_lf": self.adstock_lf.to_dict(),
            "saturation_lf": self.saturation_lf.to_dict(),
        }

    def incrementality_spec(self) -> IncrementalitySpec:
        """Return incrementality settings for this effect."""
        return IncrementalitySpec()

    def create_effect(self, mmm):
        """Build the intro funnel DAG inside the MMM graph."""
        model = mmm.model
        upper = mmm.channel_data_scaled.isel(channel=0)
        baseline = pmd.HalfNormal(f"{self.prefix}_baseline", sigma=1.0)
        gamma = pmd.HalfNormal(f"{self.prefix}_gamma", sigma=1.0)
        upper_on_lower = self.saturation_uf.apply(
            self.adstock_uf.apply(upper, core_dim="date"), core_dim="date"
        )
        demand = pmd.Deterministic(
            f"{self.prefix}_lower_demand",
            baseline + gamma * model["lower_control"] + upper_on_lower,
        )
        pmd.TruncatedNormal(
            f"{self.prefix}_lower_likelihood",
            mu=demand,
            sigma=pmd.HalfNormal(f"{self.prefix}_sigma", sigma=1.0),
            lower=0.0,
            observed=model["lower_spend"],
        )
        return pmd.Deterministic(
            f"{self.prefix}_effect_contribution",
            self.saturation_lf.apply(
                self.adstock_lf.apply(demand, core_dim="date"), core_dim="date"
            ),
        )


class GeoFunnelEffect(DataVarMuEffect):
    """Advanced geo-panel funnel effect (ported from mmm_funnel_mueffect_advanced)."""

    upper_transform: InstanceOf[MediaTransformation]
    demand_transform: InstanceOf[MediaTransformation]

    model_config = {"arbitrary_types_allowed": True}

    def to_dict(self) -> dict:
        """Serialize effect configuration for model persistence."""
        return {
            "data_vars": self.data_vars,
            "prefix": self.prefix,
            "upper_transform": self.upper_transform.to_dict(),
            "demand_transform": self.demand_transform.to_dict(),
        }

    def incrementality_spec(self) -> IncrementalitySpec:
        """Return incrementality settings including lower-funnel carryover."""
        return IncrementalitySpec(
            additional_carryover_lags=self.demand_transform.adstock.l_max
        )

    def create_effect(self, mmm):
        """Build the geo funnel DAG inside the MMM graph."""
        model = mmm.model
        upper_on_demand = self.upper_transform(mmm.channel_data_scaled, dim="date").sum(
            dim="channel"
        )
        baseline = pmd.HalfNormal(f"{self.prefix}_baseline", sigma=1.0, dims=("geo",))
        gamma = pmd.HalfNormal(f"{self.prefix}_gamma", sigma=1.0, dims=("geo",))
        demand = pmd.Deterministic(
            f"{self.prefix}_demand",
            baseline + gamma * model["category_demand"] + upper_on_demand,
        )
        lam_b = pmd.HalfNormal(f"{self.prefix}_lambda", sigma=0.5, dims=("geo",))
        lf_spend = pmd.Deterministic(
            f"{self.prefix}_lf_spend",
            demand + lam_b * model["lf_budget"],
        )
        pmd.TruncatedNormal(
            f"{self.prefix}_lower_spend_likelihood",
            mu=lf_spend,
            sigma=pmd.HalfNormal(f"{self.prefix}_sigma_m", sigma=1.0, dims=("geo",)),
            lower=0.0,
            observed=model["lower_spend"],
        )
        kappa = pmd.HalfNormal(f"{self.prefix}_kappa", sigma=1.0)
        pmd.Normal(
            f"{self.prefix}_search_likelihood",
            mu=kappa * demand,
            sigma=pmd.HalfNormal(f"{self.prefix}_sigma_s", sigma=1.0, dims=("geo",)),
            observed=model["search_volume"],
        )
        return pmd.Deterministic(
            f"{self.prefix}_effect_contribution",
            self.demand_transform(lf_spend, dim="date"),
        )


REGISTRY["FunnelEffect"] = FunnelEffect
REGISTRY["GeoFunnelEffect"] = GeoFunnelEffect

## Part 1 — Intro funnel (national)

In [ ]:
n_dates = 130
l_max = 8
date_range = pd.date_range(start="2021-01-04", freq="W-MON", periods=n_dates)

t = np.arange(n_dates) / n_dates
cov_coords = {"date": date_range, "driver": ["upper_spend", "lower_control"]}
with pm.Model(coords=cov_coords) as covariates_model:
    t_data = pm.Data("t", t, dims=("date",))
    L, _, _ = pm.LKJCholeskyCov("L", n=2, eta=100, sd_dist=pm.Exponential.dist(lam=1.0))
    a = pm.Normal("a", mu=0, sigma=1, dims="driver")
    b = pm.Normal("b", mu=0, sigma=1, dims="driver")
    mu_cov = pm.Deterministic(
        "mu_cov", a + b * t_data[..., None], dims=("date", "driver")
    )
    x_raw = pm.MvNormal("x_raw", mu=mu_cov, chol=L, dims=("date", "driver"))
    x = pm.Deterministic("x", pt.softplus(x_raw), dims=("date", "driver"))

x_data = pm.draw(covariates_model.x, draws=1, random_seed=rng)
df_intro = pd.DataFrame(
    {
        "date": date_range,
        "upper_spend": x_data[:, 0],
        "lower_control": x_data[:, 1],
        "lower_spend": np.ones(n_dates),
        "y_dummy": np.ones(n_dates),
    }
)

true_intro = {
    "intercept_contribution": 0.25,
    "adstock_alpha": 0.45,
    "saturation_lam": 4.0,
    "saturation_beta": 0.80,
    "y_sigma": 0.05,
    "funnel_baseline": 0.15,
    "funnel_gamma": 0.25,
    "adstock_uf_alpha": 0.50,
    "sat_uf_lam": 2.5,
    "sat_uf_beta": 1.3,
    "funnel_sigma": 0.04,
    "adstock_lf_alpha": 0.30,
    "sat_lf_lam": 1.2,
    "sat_lf_beta": 1.1,
}
true_intro = {k: np.asarray(v, dtype=float) for k, v in true_intro.items()}

In [ ]:
def make_intro_dataset(frame: pd.DataFrame) -> xr.Dataset:
    dates = pd.to_datetime(frame["date"]).to_numpy()
    return xr.Dataset(
        {
            "media": xr.DataArray(
                frame[["upper_spend"]].to_numpy(dtype=float),
                dims=("date", "channel"),
                coords={"date": dates, "channel": ["upper_spend"]},
            ),
            "lower_spend": xr.DataArray(
                frame["lower_spend"].to_numpy(dtype=float),
                dims=("date",),
                coords={"date": dates},
            ),
            "lower_control": xr.DataArray(
                frame["lower_control"].to_numpy(dtype=float),
                dims=("date",),
                coords={"date": dates},
            ),
        }
    )


model_config_gen = {
    "intercept": Prior("Normal", mu=0.3, sigma=0.1),
    "likelihood": Prior("Normal", sigma=Prior("HalfNormal", sigma=0.1)),
}
gen_intro = build_mmm_from_yaml(
    CONFIG_DIR / "funnel_intro.yml",
    X=make_intro_dataset(df_intro),
    y=df_intro["y_dummy"],
    model_kwargs={
        "target_column": "y_dummy",
        "model_config": model_config_gen,
    },
)
channel_dim_vars = {"adstock_alpha", "saturation_lam", "saturation_beta"}
intervention = {
    k: (true_intro[k].reshape(1) if k in channel_dim_vars else true_intro[k])
    for k in true_intro
}
gen_intro.model = do(gen_intro.model, intervention)
with gen_intro.model:
    idata_gen = pm.sample_prior_predictive(
        draws=1,
        var_names=[
            "y_original_scale",
            "channel_contribution_original_scale",
            "funnel_effect_contribution_original_scale",
            "funnel_lower_likelihood",
        ],
        random_seed=rng,
    )
prior = idata_gen["prior"].sel(chain=0, draw=0)
prior_pp = idata_gen["prior_predictive"].sel(chain=0, draw=0)
df_intro["lower_spend"] = prior_pp["funnel_lower_likelihood"].to_numpy().ravel()
df_intro["y_obs"] = prior["y_original_scale"].to_numpy().ravel()
df_intro = df_intro.tail(-l_max).reset_index(drop=True)

In [ ]:
sample_kwargs = dict(chains=2, tune=500, draws=500, target_accept=0.9, random_seed=rng)
X_intro = df_intro[["date", "upper_spend", "lower_spend"]]
X_funnel_intro = make_intro_dataset(df_intro)
y_intro = df_intro["y_obs"]

intro_specs = {
    "Naive A": CONFIG_DIR / "funnel_intro_naive_a.yml",
    "Naive B": CONFIG_DIR / "funnel_intro_naive_b.yml",
    "Funnel": CONFIG_DIR / "funnel_intro.yml",
}
intro_models = {}
for label, path in intro_specs.items():
    X = X_funnel_intro if label == "Funnel" else X_intro
    m = build_mmm_from_yaml(path, X=X, y=y_intro)
    m.fit(X=X_intro if label == "Funnel" else X, y=y_intro, **sample_kwargs)
    intro_models[label] = m

In [ ]:
def upper_roas(model):
    roas = model.incrementality.contribution_over_spend(frequency="all_time")
    return roas.sel(channel="upper_spend")


intro_roas = {label: float(upper_roas(m).mean()) for label, m in intro_models.items()}
pd.Series(intro_roas, name="upper-funnel ROAS").to_frame().T

## Part 2 — Advanced funnel (geo panel)

In [ ]:
geos = ["north", "south", "west"]
channels = ["tv_spend", "social_spend"]
n_dates_adv = 130
l_max = 8
date_range_adv = pd.date_range(start="2021-01-04", freq="W-MON", periods=n_dates_adv)
coords_adv = {"date": date_range_adv, "geo": geos}

t_adv = np.arange(n_dates_adv) / n_dates_adv
week_of_year = date_range_adv.isocalendar().week.to_numpy() / 52.0
season = np.sin(2 * np.pi * week_of_year)
drivers = ["tv_spend", "social_spend", "category_demand"]
geo_size = np.array([1.0, 0.7, 0.45])
level = np.array([0.10, 0.05, 0.30])
trend = np.array([1.20, 0.15, 1.00])
seasonal_amplitude = np.array([0.80, 0.40, 0.70])
cov_coords_adv = {"date": date_range_adv, "geo": geos, "driver": drivers}
with pm.Model(coords=cov_coords_adv) as covariates_model_adv:
    L, _, _ = pm.LKJCholeskyCov(
        "L", n=3, eta=50, sd_dist=pm.Gamma.dist(mu=0.30, sigma=0.05)
    )
    mu_cov = (
        level
        + trend * t_adv[:, None, None]
        + seasonal_amplitude * season[:, None, None]
    )
    x_raw = pm.MvNormal("x_raw", mu=mu_cov, chol=L, dims=("date", "geo", "driver"))
    x = pm.Deterministic("x", pt.softplus(x_raw), dims=("date", "geo", "driver"))

x_adv = (
    pm.draw(covariates_model_adv.x, draws=1, random_seed=rng) * geo_size[None, :, None]
)
media_raw = x_adv[..., :2]
category_demand = x_adv[..., 2] / x_adv[..., 2].max()
lf_budget = 0.15 * category_demand

In [ ]:
def make_advanced_dataset(
    media, lower_spend, search_volume, cat_demand, budget
) -> xr.Dataset:
    return xr.Dataset(
        {
            "media": xr.DataArray(media, dims=("date", "geo", "channel")),
            "lower_spend": xr.DataArray(lower_spend, dims=("date", "geo")),
            "search_volume": xr.DataArray(search_volume, dims=("date", "geo")),
            "category_demand": xr.DataArray(cat_demand, dims=("date", "geo")),
            "lf_budget": xr.DataArray(budget, dims=("date", "geo")),
        },
        coords={**coords_adv, "channel": channels},
    )


true_adv = {
    "intercept_contribution": np.array([0.35, 0.28, 0.22]),
    "adstock_alpha": np.array([0.55, 0.30]),
    "saturation_lam": np.array([3.0, 4.0]),
    "saturation_beta": np.array([[0.55, 0.35], [0.45, 0.30], [0.35, 0.25]]),
    "gamma_fourier": np.array(
        [
            [0.05, 0.03, 0.02, -0.02],
            [0.04, 0.02, 0.02, -0.01],
            [0.03, 0.02, 0.01, -0.01],
        ]
    ),
    "y_sigma": np.array([0.04, 0.04, 0.04]),
    "funnel_baseline": np.array([0.20, 0.16, 0.12]),
    "funnel_gamma": np.array([0.35, 0.30, 0.25]),
    "adstock_uf_alpha": np.array([0.60, 0.35]),
    "sat_uf_lam": np.array([2.5, 3.0]),
    "sat_uf_beta": np.array([[0.85, 0.55], [0.70, 0.45], [0.55, 0.35]]),
    "funnel_lambda": np.array([0.50, 0.45, 0.40]),
    "funnel_sigma_m": np.array([0.05, 0.05, 0.05]),
    "funnel_kappa": np.array(0.8),
    "funnel_sigma_s": np.array([0.05, 0.05, 0.05]),
    "adstock_lf_alpha": np.array([0.35, 0.30, 0.25]),
    "sat_lf_lam": np.array(1.5),
    "sat_lf_beta": np.array([1.05, 0.90, 0.72]),
}

ones = np.ones((n_dates_adv, len(geos)))
ds_gen = make_advanced_dataset(media_raw, ones, ones, category_demand, lf_budget)
y_gen = xr.DataArray(ones, dims=("date", "geo"), coords=coords_adv)

gen_adv = build_mmm_from_yaml(CONFIG_DIR / "funnel_advanced.yml", X=ds_gen, y=y_gen)
gen_adv.model = do(gen_adv.model, true_adv)
pp_names = ["funnel_lower_spend_likelihood", "funnel_search_likelihood"]
var_names = [
    "y_original_scale",
    "channel_contribution_original_scale",
    "funnel_effect_contribution_original_scale",
    "funnel_demand",
    "funnel_lf_spend",
]
with gen_adv.model:
    idata_gen_adv = pm.sample_prior_predictive(
        draws=1, var_names=var_names + pp_names, random_seed=rng
    )
prior_adv = idata_gen_adv["prior"].sel(chain=0, draw=0)
prior_pp_adv = idata_gen_adv["prior_predictive"].sel(chain=0, draw=0)

y_obs = prior_adv["y_original_scale"].transpose("date", "geo").to_numpy()
lower_spend_obs = (
    prior_pp_adv["funnel_lower_spend_likelihood"].transpose("date", "geo").to_numpy()
)
search_obs = (
    prior_pp_adv["funnel_search_likelihood"].transpose("date", "geo").to_numpy()
)
y_build = xr.DataArray(y_obs, dims=("date", "geo"), coords=coords_adv)
ds_fit = make_advanced_dataset(
    media_raw, lower_spend_obs, search_obs, category_demand, lf_budget
)

rows = []
for i, date in enumerate(date_range_adv):
    for j, geo in enumerate(geos):
        rows.append(
            {
                "date": date,
                "geo": geo,
                "tv_spend": media_raw[i, j, 0],
                "social_spend": media_raw[i, j, 1],
                "category_demand": category_demand[i, j],
                "t": t_adv[i],
                "y": y_obs[i, j],
            }
        )
df_adv = pd.DataFrame(rows)
y_fit = df_adv["y"]

In [ ]:
df_naive_a = df_adv.copy()
df_naive_a["lower_spend"] = lower_spend_obs.reshape(-1)
df_naive_c = df_adv.copy()
df_naive_c["search_volume"] = search_obs.reshape(-1)

advanced_specs = {
    "Naive A": (CONFIG_DIR / "funnel_advanced_naive_a.yml", df_naive_a),
    "Naive B": (CONFIG_DIR / "funnel_advanced_naive_b.yml", df_adv),
    "Naive B+": (CONFIG_DIR / "funnel_advanced_naive_bplus.yml", df_adv),
    "Naive B++": (CONFIG_DIR / "funnel_advanced_naive_bpp.yml", df_adv),
    "Naive C": (CONFIG_DIR / "funnel_advanced_naive_c.yml", df_naive_c),
    "Funnel": (CONFIG_DIR / "funnel_advanced.yml", df_adv),
}
advanced_models = {}
for label, (path, frame) in advanced_specs.items():
    X_frame = frame.drop(columns=["y"])
    X_build = ds_fit if label == "Funnel" else X_frame
    m = build_mmm_from_yaml(path, X=X_build, y=y_build)
    m.fit(X=X_frame, y=y_fit, **sample_kwargs)
    advanced_models[label] = m

In [ ]:
def tv_roas(model):
    roas = model.incrementality.contribution_over_spend(frequency="all_time")
    return float(roas.sel(channel="tv_spend").mean())


advanced_roas = {label: tv_roas(m) for label, m in advanced_models.items()}
pd.Series(advanced_roas, name="TV ROAS (incrementality)").to_frame().T

## Summary

- YAML `model:` blocks cover naive MMMs; `effects:` + `extra_vars` cover funnel structure.
- `build_mmm_from_yaml` converts DataFrames with `extra_vars` to `xr.Dataset` automatically.
- `FunnelEffect` remains notebook-local for now; promoting it to library code is the next step.


In [ ]:
%load_ext watermark
%watermark -n -u -v -iv -w -p pymc_marketing,pytensor